# Customer Lifetime Value (CLV) Prediction

### Customer Experience Churn Analytics — Banking-AI

This notebook explains how banks estimate the total value a customer will bring over their lifetime:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of customer experience and churn analytics terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the customer experience and churn analytics prediction task and why regression modelling supports this banking decision.
- Implement a regression workflow and evaluate predictive accuracy using domain-appropriate error metrics.
- Evaluate whether the prediction error is acceptable for the operational decision it supports.

**Estimated time:** 35–45 minutes

**Collection context:** Customer churn prediction, segmentation, lifetime value, and retention strategy in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Not all customers are equal. Some maintain high balances and pay many fees, while others might cost more to support than they bring in. By predicting Customer Lifetime Value (CLV), banks can decide how much to spend on acquiring a new customer (Customer Acquisition Cost) and who should get "VIP" treatment.

**Input data used:** Monthly transactions, average balance, number of products, tenure, and referral count.

**What we predict:** Estimated Lifetime Value in Dollars (`CLV`).

**ML method used:** Random Forest Regressor.

**Learning goal:** Learn how to perform regression to predict a continuous dollar amount.

**Primary stakeholders:** relationship managers, retention teams, marketing analysts, and branch managers.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic CLV Dataset

We simulate 5,000 customers and their financial behavior.

In [ ]:
n = 5000
rng = RNG

tenure_years = rng.uniform(1, 15, n)
avg_monthly_transactions = rng.poisson(20, n)
avg_balance = rng.normal(5000, 3000, n).clip(100, 50000)
num_products = rng.choice([1, 2, 3, 4, 5], n, p=[0.4, 0.3, 0.15, 0.1, 0.05])
referrals = rng.poisson(0.5, n)

# Logic for CLV: High balance, high transaction count, and more products increase value.
# Fees + Interest Margin + Referral Value
clv = (
    (avg_balance * 0.02) + 
    (avg_monthly_transactions * 5) + 
    (num_products * 200) + 
    (referrals * 500) +
    rng.normal(0, 100, n)
) * tenure_years

df = pd.DataFrame({
    'tenure_years': tenure_years,
    'avg_monthly_transactions': avg_monthly_transactions,
    'avg_balance': avg_balance,
    'num_products': num_products,
    'referrals': referrals,
    'clv': clv
})

print(f"Average CLV: ${df['clv'].mean():.2f}")
df.head()

## Step 4 — Exploratory Data Analysis

EDA

How does Balance and Tenure impact the Lifetime Value?

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x='avg_balance', y='clv', alpha=0.5)
plt.title('Balance vs. CLV')

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='num_products', y='clv')
plt.title('CLV by Number of Products')

plt.tight_layout()
plt.show()

**Alt-text:** Distribution plot, scatter plot titled "Balance vs. CLV" and "CLV by Number of Products". The chart highlights distributional differences between groups that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('clv', axis=1)
y = df['clv']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the training-set mean ---
baseline_reg = DummyRegressor(strategy='mean')
baseline_reg.fit(X_train, y_train)
baseline_pred_bl = baseline_reg.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred_bl))
print(f"Baseline RMSE (predict-mean): {baseline_rmse:.4f}")
print("Any useful model must produce a lower RMSE.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Mean Absolute Error: ${mean_absolute_error(y_test, y_pred):.2f}")
print(f"R2 Score (Variance Explained): {r2_score(y_test, y_pred):.2f}")

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', color='darkgreen')
plt.title('Drivers of Customer Lifetime Value')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "Drivers of Customer Lifetime Value". The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- **Tenure** and **Average Balance** are the primary drivers of CLV in our model.
- Customers with more **Products** and **Referrals** significantly increase the bank's long-term profitability.

**Banking Context:** The marketing department can use this model to find customers who have low current CLV but look like "High-Value" customers (e.g., young professionals with high growth potential) and target them with loyalty programs before they become valuable to competitors.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: inspect predictions at different points ---
print("Operational demonstration — model predictions across the test range:\n")
low_idx  = int(np.argmin(y_test.values))
high_idx = int(np.argmax(y_test.values))
mid_idx  = int(np.argsort(y_test.values)[len(y_test) // 2])

for label, idx in [("Low-end", low_idx), ("Mid-range", mid_idx), ("High-end", high_idx)]:
    actual = y_test.values[idx]
    pred   = y_pred[idx]
    print(f"  {label}  actual: {actual:.4f}  predicted: {pred:.4f}  error: {abs(actual-pred):.4f}")

print("\nPractitioners would use these predictions as one input among several.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end customer experience and churn analytics workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Churn models that rely on demographic features risk discriminatory retention offers.
- Aggressive retention tactics driven by model outputs may erode customer trust.
- Customers should benefit from personalisation, not be manipulated by it.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in customer experience and churn analytics?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.